# FLATUP 動画スタジオ — MiniMax H3 画像→動画（Google Colab + ComfyUI 公式ワークフロー）[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatup1/flatup-ai-os/blob/main/notebooks/colab_minimax_h3_i2v.ipynb)**ちびキャラの静止画1枚 → 6秒の縦動画（音つき）** を作るノートブックです。ColabのGPUに **ComfyUI** を立てて、**ComfyUI公式の MiniMax H3 I2V テンプレート**をそのまま使います。## ⚠️ 先に読む（いちばん大事）MiniMax H3 のオープンウェイトは **とても大きい**（I2V一式でおよそ42GB）。そのため **無料ColabのT4（15GB）では動きません**。| GPU | VRAM | 判定 ||---|---|---|| A100 80GB / H100 | 80GB | ◎ 余裕 || A100 40GB（Colab Pro+） | 40GB | ○ 動く（int8＋省メモリ設定） || L4 | 24GB | △ かなり厳しい（落ちやすい） || T4（無料） | 15GB | ✕ 動かない |**動かないGPUのときは、このノートは途中で止まります**（時間とお金を無駄にしないため）。そのときの代わりの道は2つ:1. **今すぐ動く道** → リポジトリの `npm run reel`（fal.ai の Hailuo I2V。1本あたり数十円、環境構築ゼロ）2. **ComfyUIのまま使う道** → 公式の **MiniMax API テンプレート**（重みを落とさずAPIで生成）くわしい比較は `docs/minimax_h3_colab.md` を見てください。## 使い方（3ステップ）1. **ランタイム → ランタイムのタイプを変更 → A100** にする2. 上から順にセルを1回ずつ実行する（①〜⑥）3. 出てきた公開URLでComfyUIを開く。または ⑦〜⑨ で **このノートだけで自動生成**する> このノートは「下書き用の素材づくり」です。SNS投稿・広告出稿・お客様への送信は> かならず人間（JIN）が最終確認してから行ってください。

---## ① GPU とディスクを確認する

In [ ]:
# ① GPU / ディスク / ランタイムの確認（ここで「無理なGPU」を先に見つける）import shutil, subprocess, sysdef sh(cmd):    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()name = sh("nvidia-smi --query-gpu=name --format=csv,noheader") or "(GPUなし)"vram_mb = sh("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits")VRAM_GB = round(int(vram_mb) / 1024) if vram_mb.isdigit() else 0DISK_FREE_GB = round(shutil.disk_usage("/content").free / 1e9)print(f"GPU        : {name}")print(f"VRAM       : {VRAM_GB} GB")print(f"空きディスク: {DISK_FREE_GB} GB")# Blackwell(RTX 50xx / B200)だけ NVFP4 が速い。それ以外は int8 を選ぶ判断に使う。IS_BLACKWELL = any(k in name.upper() for k in ("B200", "B100", "RTX 50", "GB200"))if VRAM_GB >= 70:    VERDICT, HINT = "◎ 余裕", "そのまま進めてください。"elif VRAM_GB >= 38:    VERDICT, HINT = "○ 動く", "int8モデル＋省メモリ起動で進めます。"elif VRAM_GB >= 22:    VERDICT, HINT = "△ 厳しい", "途中で落ちる可能性が高いです。落ちたら A100 に変えてください。"else:    VERDICT, HINT = "✕ 動かない", "ランタイムのタイプを A100 に変えてから、もう一度①から実行してください。"print(f"\n判定       : {VERDICT}\n{HINT}")if VRAM_GB < 22:    print("\n" + "=" * 60)    print("このGPUでは MiniMax H3 のオープンウェイトは動きません。")    print("代わりの道:")    print("  1) リポジトリの `npm run reel`（fal.ai Hailuo I2V / 環境構築ゼロ）")    print("  2) ComfyUI公式の MiniMax API テンプレート（重みを落とさない）")    print("=" * 60)if DISK_FREE_GB < 60:    print("\n⚠ ディスクの空きが少なめです（モデル一式でおよそ42GB使います）。")

---## ② 保存先（Google Drive）をつなぐ- **出力の動画は必ずDriveに保存**します（Colabのセッションが切れると消えるため）- **モデル（約42GB）をDriveに置くかは選べます**  - Drive無料は15GBなので **入りません**。100GB以上のプランのときだけ `True` にしてください  - `False` のときはセッションのディスクに落とします（毎回ダウンロードが必要）

In [ ]:
# ② Google Drive をつなぐMODELS_ON_DRIVE = False   # Driveが100GB以上あるなら True（毎回のDLが不要になる）from google.colab import driveimport osdrive.mount('/content/drive')DRIVE_ROOT = '/content/drive/MyDrive/FLATUP_H3'OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'      # 完成した動画（必ずDrive）os.makedirs(OUTPUT_DIR, exist_ok=True)MODEL_DIR = f'{DRIVE_ROOT}/models' if MODELS_ON_DRIVE else '/content/models'os.makedirs(MODEL_DIR, exist_ok=True)print('動画の保存先 :', OUTPUT_DIR)print('モデルの置場 :', MODEL_DIR, '(Drive)' if MODELS_ON_DRIVE else '(セッション内・毎回消える)')

---## ③ ComfyUI を入れる（3〜5分）公式リポジトリをそのまま入れます。MiniMax H3 は新しいComfyUIに入っている機能なので、**最新（master）** を使います。

In [ ]:
# ③ ComfyUI 本体 + 公開URL用の cloudflared を入れるimport os, subprocessCOMFY = '/content/ComfyUI'if not os.path.exists(COMFY):    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}else:    print('取得済みなのでスキップ')# torch は Colab のものをそのまま使う（requirements.txt は torch を固定していない）!pip -q install -r {COMFY}/requirements.txt# モデル置き場を MODEL_DIR に向ける（ComfyUI公式の extra_model_paths.yaml を使う）#   ※ シンボリックリンクでも動くが、ComfyUI側の既定フォルダに中身があると#      リンクが張れず「モデルが見つからない」になる。公式の仕組みのほうが確実。for sub in ('diffusion_models', 'text_encoders', 'vae'):    os.makedirs(f'{MODEL_DIR}/{sub}', exist_ok=True)with open(f'{COMFY}/extra_model_paths.yaml', 'w') as f:    f.write(f"""flatup:    base_path: {MODEL_DIR}    diffusion_models: diffusion_models    text_encoders: text_encoders    vae: vae""")print('モデル置き場を登録しました:', MODEL_DIR)# 公開URLを作る道具（ComfyUIの画面をブラウザで開くため）if not os.path.exists('/usr/local/bin/cloudflared'):    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64    !chmod +x /usr/local/bin/cloudflared!pip -q install huggingface_hub hf_transferprint('ComfyUI の準備ができました')

---## ④ 使うモデルを決めるComfy-Org の公式配布（`Comfy-Org/MiniMax-H3`）から、**ComfyUI公式テンプレートと同じ4つ**を選びます。| 置き場所 | 役割 ||---|---|| `diffusion_models/` | 動画を描く本体（I2V/T2V共通の fl2va） || `text_encoders/` | 文章と画像を読む係（Qwen3-VL-32B） || `vae/` | 映像のVAE || `vae/` | 音声のVAE（H3は音も一緒に作る） |text encoder は **GPUに合わせて自動で選びます**（Blackwell以外は int8）。

In [ ]:
# ④ モデルの一覧を見て、使う4ファイルを決めるfrom huggingface_hub import HfApiREPO = 'Comfy-Org/MiniMax-H3'api = HfApi()info = api.model_info(REPO, files_metadata=True)files = {s.rfilename: (s.size or 0) for s in info.siblings}print('=== 配布ファイル一覧 ===')for f, sz in sorted(files.items()):    if f.endswith('.safetensors'):        print(f'{sz/1e9:6.2f} GB  {f}')def pick(folder, prefer):    """folder内から prefer(部分一致リスト)の順で最初に見つかったものを返す"""    cands = [f for f in files if f.startswith(folder + '/') and f.endswith('.safetensors')]    for key in prefer:        for f in cands:            if key in f:                return f    return cands[0] if cands else None# 本体: I2V/T2V共通の fl2va。省メモリの pruned int8 を優先（公式テンプレの既定と同じ）UNET = pick('diffusion_models', ['fl2va_pruned_int8', 'fl2va_int8', 'fl2va'])# 文章係: Blackwell(RTX50/B200)だけ NVFP4が速い。それ以外は int8。TE_PREF = ['nvfp4', 'int8', 'bf16'] if IS_BLACKWELL else ['int8', 'nvfp4', 'bf16']CLIP = pick('text_encoders', TE_PREF)VAE_VIDEO = pick('vae', ['video_vae_fp16', 'video_vae'])VAE_AUDIO = pick('vae', ['audio_vae_fp32', 'audio_vae'])CHOSEN = [UNET, CLIP, VAE_VIDEO, VAE_AUDIO]total = sum(files.get(f, 0) for f in CHOSEN) / 1e9print('\n=== これをダウンロードします ===')for f in CHOSEN:    print(f'{files.get(f,0)/1e9:6.2f} GB  {f}')print(f'合計 {total:.1f} GB')if total > DISK_FREE_GB - 5:    print('\n⚠ ディスクが足りない可能性があります。MODELS_ON_DRIVE や不要ファイルを見直してください。')

---## ⑤ モデルをダウンロード（初回15〜40分）**途中で止まっても、もう一度実行すれば続きから**落とせます（同じファイルは再取得しません）。

In [ ]:
# ⑤ モデルをダウンロード（再実行で続きから）import os, timeos.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'from huggingface_hub import hf_hub_downloadfor f in CHOSEN:    folder, fname = f.split('/', 1)    dest = f'{MODEL_DIR}/{folder}/{fname}'    known = files.get(f, 0)   # 配布側がサイズを返さないことがある（0のときは存在だけ見る）    if os.path.exists(dest) and (os.path.getsize(dest) == known or                                 (known == 0 and os.path.getsize(dest) > 0)):        print(f'済み: {fname}')        continue    print(f'取得中: {fname} ({files.get(f,0)/1e9:.1f} GB)')    t0 = time.time()    p = hf_hub_download(REPO, f, local_dir=MODEL_DIR)    # local_dir に repo と同じ階層で落ちるので、そのまま使える    print(f'  完了 {(time.time()-t0)/60:.1f}分  -> {p}')print('\n=== 置かれたファイル ===')for sub in ('diffusion_models', 'text_encoders', 'vae'):    for fn in sorted(os.listdir(f'{MODEL_DIR}/{sub}')):        sz = os.path.getsize(f'{MODEL_DIR}/{sub}/{fn}') / 1e9        print(f'{sz:6.2f} GB  {sub}/{fn}')UNET_NAME = UNET.split('/', 1)[1]CLIP_NAME = CLIP.split('/', 1)[1]VAE_VIDEO_NAME = VAE_VIDEO.split('/', 1)[1]VAE_AUDIO_NAME = VAE_AUDIO.split('/', 1)[1]

---## ⑥ ComfyUI を起動して、公開URLを出す出てきた `https://....trycloudflare.com` を開くと、いつものComfyUIの画面が出ます。

In [ ]:
# ⑥ ComfyUI を起動（バックグラウンド）＋ 公開URLを作るimport subprocess, time, re, socket, osos.makedirs('/content/comfy_logs', exist_ok=True)LOG = '/content/comfy_logs/comfyui.log'TUNNEL_LOG = '/content/comfy_logs/tunnel.log'def port_open(p):    with socket.socket() as s:        s.settimeout(1)        return s.connect_ex(('127.0.0.1', p)) == 0if not port_open(8188):    # --cache-none: 1回ごとにメモリを解放（H3は重いので落ちにくくする）    cmd = f'python {COMFY}/main.py --listen 127.0.0.1 --port 8188 --cache-none --disable-auto-launch'    subprocess.Popen(f'{cmd} > {LOG} 2>&1', shell=True)for i in range(180):    if port_open(8188):        print('ComfyUI 起動OK')        break    time.sleep(2)else:    print('起動できませんでした。ログの最後を見てください:')    !tail -30 {LOG}subprocess.Popen(    f'cloudflared tunnel --url http://127.0.0.1:8188 --no-autoupdate > {TUNNEL_LOG} 2>&1', shell=True)url = Nonefor i in range(60):    time.sleep(2)    try:        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open(TUNNEL_LOG).read())        if m:            url = m.group(0)            break    except FileNotFoundError:        passprint('\n公開URL:', url or '（作れませんでした。⑦以降の自動生成はURL無しでも動きます）')print("""--- 画面での使い方 ---1. 上のURLを開く2. メニュー Workflow -> Browse Templates -> Video3. 「MiniMax H3: Image to Video」を選ぶ（これが公式テンプレート）4. LoadImage にちびキャラ画像を入れる5. prompt を ⑦ で作ったものに差し替える6. Resolution Selector を 9:16 にして Run""")

---## ⑦ FLATUP用のテンプレプロンプト（差し替え方式）H3は **映像と音を同時に**作ります。だから文章は次の順番で書くのがコツです。```世界観（毎回同じ） → タイムライン（何秒に何が起きる） → 音（Audio:） → 禁止事項```使いたい場面の名前を `SCENE` に入れて実行するだけで、完成プロンプトが出ます。**キャラの出典は2つあります。混ぜないでください。**| 出典 | キャラ | 使いどころ ||---|---|---|| `docs/flatup_animation_bible.md`（v3.0・シリーズ本編の正本） | **マサキ**（19歳・指導役） / **ツム**（5歳・主人公） | EP1〜10の本編カット || `docs/flatup_anime_studio.md`（2026-07-16 JIN確定） | **あぷちゃん**（マスコット） / ミットくん | 単発のSNSリール |姿かたちの英文は `src/reel/characters.ts` の `look` と同じ文にしてあるので、`npm run img` / `npm run reel` で作った画像とブレません。禁止事項はバイブルの「描いてはいけないもの」をそのまま入れてあります。

In [ ]:
# ⑦ FLATUP テンプレプロンプト（ここだけ差し替えて使う）SCENE   = 'first_punch'   # 下の一覧から選ぶ（最初の1本はこれがおすすめ）SECONDS = 6               # 6秒（H3は最大15秒くらいまで）# --- 世界観（毎回同じ。ここを固定するとブレない） -------------------------# 出典は2つある。混ぜないこと。#   A) ANIMATION BIBLE v3.0（docs/flatup_animation_bible.md）= シリーズ本編の正本#      → マサキ / ツム など。姿かたちは src/reel/characters.ts と同じ文にしてある#   B) あぷちゃん（docs/flatup_anime_studio.md・2026-07-16 JIN確定）= SNSリール用マスコット#      → バイブル本編のキャストではない。単発リール用として使うWORLD_APU = (    "Warm 3D animation movie style, bright and gentle mood. "    "A cute 2.5-head-tall chibi girl character with big brown eyes and a tall strong dark brown ponytail, "    "wearing a pink heart-pattern T-shirt and pink-and-black muay thai shorts, oversized red boxing gloves, barefoot. "    "Setting: a bright clean gym with a white floor, a big pink star circle on the floor, potted plants, "    "a yellow punching bag and a neon pink punching bag, soft natural light. The environment is constant throughout.")# マサキ（19歳・キックボクシングインストラクター / ANIMATION BIBLE v3.0 のメインキャスト）# 姿かたちは src/reel/characters.ts の look をそのまま使う（実装側の正本と一致させる）WORLD_MASAKI = (    "Warm 3D animation movie style, bright and gentle mood. "    "A friendly young male kickboxing coach in his late teens with short black hair, "    "black FLAT UP GYM hoodie, warm encouraging smile, stylized 3D animation character. "    "Setting: a bright clean gym with a white floor and green mats, soft warm light. The environment is constant throughout.")# ツム（5歳・シリーズの主人公 / ANIMATION BIBLE v3.0）。最初は怖がりで、少しずつ成長するWORLD_TSUMU = (    "Warm 3D animation movie style, bright and gentle mood. "    "A 2.5-head-tall chibi 5-year-old girl with shoulder-length light brown wavy hair in a small ponytail, "    "large round glossy brown eyes, rosy cheeks, white t-shirt and light blue FLAT UP GYM muay thai shorts, "    "barefoot, shy gentle expression. "    "Setting: a bright clean gym with a white floor and green mats, soft warm light. The environment is constant throughout.")WORLD_NIGHT = (    "Warm 3D animation movie style, quiet night mood. "    "An empty gym after closing time, moonlight through the window, a white-and-green focus mitt with big gentle eyes "    "resting on the ring apron. Deep blue shadows with a small warm lamp. The environment is constant throughout.")# --- 禁止事項（ANIMATION BIBLE v3.0「描いてはいけないもの」＋ 制作上の決まり） ---RULES = (    "No text, no logo, no subtitles, no watermark. No real people, no photorealistic humans, "    "no resemblance to any real gym member. "    "No blood, no injury, no expression of pain, no scary face, no angry shouting. "    "No contact between fighters, no sparring, no opponent knocked down, no celebrating a win over someone. "    "Single subject only. Vertical 9:16 framing, subject centered and fully inside the frame.")# --- 場面（ここを増やせば新しい型になる） ---------------------------------SCENES = {    'first_punch': {        'jp': 'はじめの一歩（緊張→笑顔→やさしいストレート／MVPの本命）',        'world': WORLD_APU,        'timeline': [            "[0s-2s] The character stands still, looking a little nervous, both red gloves held close to her chest.",            "[2s-4s] Her expression softens into a small smile, and she takes one careful step forward.",            "[4s-6s] She gently throws exactly one straight punch toward a soft yellow foam pad standing on a small "            "training stand in front of her, then brings the glove back and smiles proudly.",        ],        'audio': "Audio: one soft footstep, a light padded tap on the foam stick, warm uplifting music. No voice, no shouting.",    },    'wave': {        'jp': 'おいでおいで（体験募集リールの冒頭）',        'world': WORLD_APU,        'timeline': [            "[0s-2s] The character stands in the middle of the pink star circle, notices the camera and lights up with a big smile.",            "[2s-4s] She raises one red glove and waves hello twice, then beckons with a friendly come-here gesture.",            "[4s-6s] She hops once on the spot with both gloves up, ponytail bouncing, and holds a warm welcoming smile.",        ],        'audio': "Audio: light cheerful ukulele and soft claps, one small cute hop sound, gentle room tone. No voice.",    },    'jab': {        'jp': '左ジャブ1発（技の紹介リール）',        'world': WORLD_APU,        'timeline': [            "[0s-2s] The character settles into a relaxed guard, both red gloves up, eyes focused but friendly.",            "[2s-4s] She throws exactly one crisp left jab into the air, glove snapping forward and back to guard.",            "[4s-6s] She smiles proudly, gives a small nod, and returns to the guard stance.",        ],        'audio': "Audio: soft glove swish on the jab, a light snare-like snap, calm upbeat background beat. No voice.",    },    'high_kick': {        'jp': '元気なハイキック（動きの大きいリール）',        'world': WORLD_APU,        'timeline': [            "[0s-2s] The character takes one small step, gloves up, weight shifting onto the front foot.",            "[2s-4s] She performs one friendly high kick into the air, body turning smoothly, ponytail swinging.",            "[4s-6s] She lands lightly, wobbles once in a cute way, then grins and gives a thumbs-up with the glove.",        ],        'audio': "Audio: a light whoosh on the kick, a soft landing thud, cheerful marimba background. No voice.",    },    'rest': {        'jp': '休憩中（癒し系・保存されやすい）',        'world': WORLD_APU,        'timeline': [            "[0s-2s] The character leans against the yellow punching bag, shoulders dropping, breathing out slowly.",            "[2s-4s] Her eyes close, head tilting gently against the bag as she drifts into a short nap.",            "[4s-6s] The bag sways a few centimeters, she stirs slightly and smiles in her sleep.",        ],        'audio': "Audio: quiet gym room tone, a faint creak of the punching bag chain, slow soft piano. No voice.",    },    'masaki_welcome': {        'jp': 'マサキ先生のごあいさつ（バイブル正本キャスト）',        'world': WORLD_MASAKI,        'timeline': [            "[0s-2s] The coach turns toward the camera and smiles softly, one hand raised in a small greeting.",            "[2s-4s] He gestures to the empty mat beside him, inviting the viewer in with a calm open palm.",            "[4s-6s] He bows politely, then straightens up with a warm reassuring smile.",        ],        'audio': "Audio: warm quiet gym ambience, soft acoustic guitar, no dialogue, no shouting.",    },    'tsumu_first_step': {        'jp': 'ツムのはじめの一歩（EP1「はじめてのキックボクシング」用・バイブル主人公）',        'world': WORLD_TSUMU,        'timeline': [            "[0s-2s] The little girl stands at the edge of the green mat, holding her own hands, looking shy and unsure.",            "[2s-4s] She takes a breath, bows politely toward the mat, and steps onto it with one careful foot.",            "[4s-6s] She looks up, and a small proud smile spreads across her face as she raises both fists lightly.",        ],        'audio': "Audio: one soft footstep on the mat, quiet gym room tone, gentle warm piano rising at the end. No voice.",    },    'mitt_night': {        'jp': '夜のジムでミットくんが目を覚ます（ブランド広告EP1系）',        'world': WORLD_NIGHT,        'timeline': [            "[0s-2s] The dark empty gym is still, moonlight sliding slowly across the floor.",            "[2s-4s] The focus mitt opens its big gentle eyes, looking around at the quiet room.",            "[4s-6s] It looks toward the door where everyone left, then smiles softly and settles back down.",        ],        'audio': "Audio: quiet night room tone, a distant clock tick, a single soft piano note, gentle strings swelling at the end.",    },}def build_prompt(scene=SCENE, seconds=SECONDS):    s = SCENES[scene]    tl = "\n".join(s['timeline'])    return f"{s['world']}\n\nTimeline ({seconds} seconds total):\n{tl}\n\n{s['audio']}\n\n{RULES}"print('=== 使える場面 ===')for k, v in SCENES.items():    print(f'  {k:14s} {v["jp"]}')PROMPT = build_prompt()print('\n=== ComfyUI の prompt にこれを貼る ===\n')print(PROMPT)

---## ⑧ ちびキャラの静止画を1枚アップロード**縦長（9:16）で、キャラが真ん中に全身**で写っている画像がいちばんうまくいきます。設計シート（技が並んだ表）はそのまま入れないでください（グリッドと文字が動画に写ります）。まだ画像が無いときは、リポジトリの `npm run img -- "<英語プロンプト>" --count 4` で作れます。

In [ ]:
# ⑧ 静止画をアップロード（1枚）from google.colab import files as colab_filesfrom PIL import Imageimport os, io, shutilos.makedirs(f'{COMFY}/input', exist_ok=True)up = colab_files.upload()src_name = list(up.keys())[0]IMAGE_NAME = 'flatup_first_frame' + os.path.splitext(src_name)[1].lower()with open(f'{COMFY}/input/{IMAGE_NAME}', 'wb') as f:    f.write(up[src_name])img = Image.open(f'{COMFY}/input/{IMAGE_NAME}')ratio = img.width / img.heightprint(f'{IMAGE_NAME}  {img.width}x{img.height}  比率 {ratio:.2f}')if ratio > 0.8:    print('⚠ 横長〜正方形です。縦動画(9:16)にすると上下が引き伸ばされます。縦長の画像がおすすめ。')display(img.resize((256, int(256 / ratio))))

---## ⑨ このノートだけで6秒動画を作る（自動実行）ComfyUIの画面を触らずに、**公式テンプレートとまったく同じ流れ**をAPIで実行します。（ノードの並び・サンプラー・ステップ数は公式テンプレ `video_minimax_h3_i2v.json` と同じ）できた動画は **Driveの `FLATUP_H3/outputs/`** に保存されます。

In [ ]:
# ⑨ 自動生成（公式テンプレートと同じノード構成をAPIで流す）import json, urllib.request, urllib.parse, time, random, os, shutilASPECT  = '9:16'   # '9:16'(縦・SNS向け) / '16:9'(横) / '1:1'(正方形)STEPS   = 20       # 公式テンプレと同じSEED    = random.randint(0, 2**31)   # 数字を固定すると同じ結果を再現できるFPS = 24def adapt_canvas(ratio_w, ratio_h):    """H3の決まり: 短辺768、面積は768x1344まで、32の倍数（ComfyUI本体と同じ計算）"""    import math    BASE, MAXP, MUL = 768, 768 * 1344, 32    r = ratio_w / ratio_h    w, h = (BASE * r, BASE) if r >= 1 else (BASE, BASE / r)    if w * h > MAXP:        s = math.sqrt(MAXP / (w * h)); w, h = w * s, h * s    return max(MUL, round(w / MUL) * MUL), max(MUL, round(h / MUL) * MUL)def frames_for(seconds):    """H3は 17k+5 フレーム刻み。指定秒数以上にそろえる"""    n = max(5, round(seconds * FPS))    while n % 17 != 5:        n += 1    return nrw, rh = {'9:16': (9, 16), '16:9': (16, 9), '1:1': (1, 1)}[ASPECT]WIDTH, HEIGHT = adapt_canvas(rw, rh)LENGTH = frames_for(SECONDS)print(f'{WIDTH}x{HEIGHT} / {LENGTH}フレーム = {LENGTH/FPS:.2f}秒 / seed={SEED}')workflow = {    "6":  {"class_type": "UNETLoader",  "inputs": {"unet_name": UNET_NAME, "weight_dtype": "default"}},    "13": {"class_type": "CLIPLoader",  "inputs": {"clip_name": CLIP_NAME, "type": "minimax", "device": "default"}},    "11": {"class_type": "VAELoader",   "inputs": {"vae_name": VAE_VIDEO_NAME}},    "24": {"class_type": "VAELoader",   "inputs": {"vae_name": VAE_AUDIO_NAME}},    "114": {"class_type": "LoadImage",  "inputs": {"image": IMAGE_NAME}},    "104": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {        "clip": ["13", 0], "vae": ["11", 0], "prompt": PROMPT,        "width": WIDTH, "height": HEIGHT, "length": LENGTH,        "first_frame": ["114", 0]}},    "15": {"class_type": "RandomNoise",     "inputs": {"noise_seed": SEED}},    "17": {"class_type": "KSamplerSelect",  "inputs": {"sampler_name": "res_multistep"}},    "9":  {"class_type": "BasicScheduler",  "inputs": {"model": ["6", 0], "scheduler": "simple",                                                       "steps": STEPS, "denoise": 1.0}},    "16": {"class_type": "BasicGuider",     "inputs": {"model": ["6", 0], "conditioning": ["104", 0]}},    "14": {"class_type": "SamplerCustomAdvanced", "inputs": {        "noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],        "sigmas": ["9", 0], "latent_image": ["104", 1]}},    "10": {"class_type": "VAEDecode",       "inputs": {"samples": ["14", 0], "vae": ["11", 0]}},    "23": {"class_type": "VAEDecodeAudio",  "inputs": {"samples": ["14", 0], "vae": ["24", 0]}},    "91": {"class_type": "CreateVideo",     "inputs": {"images": ["10", 0], "audio": ["23", 0],                                                       "fps": FPS, "bit_depth": 8}},    "92": {"class_type": "SaveVideo",       "inputs": {"video": ["91", 0],                                                       "filename_prefix": f"video/FLATUP_{SCENE}",                                                       "format": "auto", "codec": "auto"}},}API = 'http://127.0.0.1:8188'def post(path, payload):    req = urllib.request.Request(API + path, data=json.dumps(payload).encode(),                                 headers={'Content-Type': 'application/json'})    return json.load(urllib.request.urlopen(req))def get(path):    return json.load(urllib.request.urlopen(API + path))try:    res = post('/prompt', {'prompt': workflow})except urllib.error.HTTPError as e:    # 400のときは「どのノードの何が悪いか」が本文に入っている。読める形で出す。    detail = e.read().decode('utf-8', 'replace')    try:        detail = json.dumps(json.loads(detail), ensure_ascii=False, indent=2)    except Exception:        pass    print('ComfyUIに拒否されました:\n', detail)    raise SystemExit('ワークフローを直してから、もう一度このセルを実行してください')pid = res['prompt_id']print('実行中… A100で6秒動画はおよそ5〜15分かかります')t0 = time.time()hist = Nonewhile True:    time.sleep(10)    h = get(f'/history/{pid}')    if pid in h:        hist = h[pid]        break    print(f'  経過 {int(time.time()-t0)}秒', end='\r')status = hist.get('status', {})if status.get('status_str') == 'error' or not status.get('completed', True):    print('\n失敗しました。ログの最後を見てください:')    for m in status.get('messages', [])[-5:]:        print(' ', m)    !tail -40 {LOG}else:    saved = []    for node_out in hist.get('outputs', {}).values():        for items in node_out.values():            if not isinstance(items, list):                continue            for it in items:                if isinstance(it, dict) and str(it.get('filename', '')).endswith(('.mp4', '.webm', '.mkv')):                    sub = it.get('subfolder', '')                    src = os.path.join(COMFY, 'output', sub, it['filename'])                    dst = os.path.join(OUTPUT_DIR, f"{SCENE}_{SEED}_{it['filename']}")                    shutil.copy(src, dst)                    saved.append(dst)    print(f"\n完成！ {int(time.time()-t0)}秒かかりました")    for s in saved:        print(' 保存:', s)    # その場で再生    if saved:        from IPython.display import HTML        import base64        b64 = base64.b64encode(open(saved[0], 'rb').read()).decode()        display(HTML(f'<video width=360 controls src="data:video/mp4;base64,{b64}"></video>'))

---## うまくいかないとき| 症状 | 直し方 ||---|---|| **メモリ不足で落ちる（OOM）** | ランタイムを A100 にする / ⑥の起動オプションに `--lowvram` を足す / ⑨の `STEPS` を 20→14 に下げる || **ダウンロードが終わらない** | ⑤をもう一度実行（続きから落ちる）。Colabのセッションが切れる前に `MODELS_ON_DRIVE=True` を検討 || **キャラの顔が変わる** | 入力画像を毎回同じ「基準画像」にする（`docs/flatup_animation_bible.md` の決まり） || **動きが破綻する（腕が増える等）** | タイムラインの動作を1カット1つに減らす。`SEED` を変えて引き直す || **画面に文字が出る** | 入力画像に文字が写っていないか確認。`RULES` の `No text` は既に入っている || **音がいらない** | 完成後にCapCut等で音を消す。または `Audio:` 行を短くする || **`MiniMaxH3ImageToVideo` が無いと言われる** | ComfyUIが古い。③のcloneをやり直す（`/content/ComfyUI` を消してから） |## つぎにやること1. **T2V（画像なし）を試す** — ⑨の workflow から `"first_frame"` の行を消すだけでテキスト→動画になります2. **R2V（参照で固定）** — 公式テンプレート「MiniMax H3: Reference to Video」でキャラ・声・動きを固定する3. **9:16で量産** — `SCENE` を変えて⑦→⑨を繰り返す。`SEED` を記録しておくと当たりを再現できる4. **投稿文まで自動化** — できた動画に対して、リポジトリの `npm run dev -- sns_post "<内容>"` でキャプションを下書きする## 大事な約束- ここで作るのは**すべて下書き**です。SNS投稿・広告・お客様への送信は**JINの確認のあと**に行います- 実在の会員さんの顔に寄せた映像は作りません（`RULES` にも入れてあります）- 料金・時間・クラス名を映像や投稿文に入れるときは、**正本**（`src/data/` と canon）を必ず確認します